# 기준선 재현 및 공통 평가

**데이터**: 연구 조가 공개 규격을 바탕으로 생성한 합성 통신망. 저장 사례의 위치는 `scenarios/problem-bank/bundles/`다. 이 노트북은 n=6·8, 생성 시드 6·7, 예산 20·40%의 8개 저장 사례를 사용한다. 저장소와 함께 실행하며 추가 데이터를 내려받지 않는다. [저장소](https://github.com/d86729/os-project-26-2) · 확인일 2026-11-24.

**방법**: 간선 매개중심성/비용 기준선과 수요 경로 기반 AI 연구 보조 전략을 각각 실행하고 공통 평가기 및 저장된 인증 참조와 비교한다. 기준선의 매개중심성은 Brandes (2001)의 계산을 참고하고 비용 정규화는 이 연구의 적용 규칙이다. [Brandes 논문](https://doi.org/10.1080/0022250X.2001.9990249) · 합성망 생성 참고 [Dong·Tian (2022)](https://doi.org/10.1155/2022/3108327).

**작성과 검토**: 공통 코드 담당은 김대건으로 기록돼 있다. 아래 노트북과 비교 전략은 AI 연구 보조 산출물이며 담당자의 검토와 설명이 필요하다.

**실행**: 프로젝트 루트에서 `pip install -e '.[test]'`로 환경을 준비하고 해당 Python 커널을 선택한다. 작성된 코드를 순서대로 모두 실행한다. 실행 결과는 앞서 저장한 점수와 다시 대조한다.

### 담당: 공통 코드 김대건 · 노트북 작성 AI

In [1]:
from pathlib import Path
import json
import subprocess
import sys
from hashlib import sha256

from protection.runner import initialize_single_thread, collect_environment
initialize_single_thread()
from protection.models import load_bundle
from protection.program import protected_ids, scenario_input
from protection.evaluation import Evaluator

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'pyproject.toml').is_file())
study = root / 'results/agent-study/2026-11-24'
protocol = json.loads((study / 'protocol.json').read_text(encoding='utf-8'))
saved = [json.loads(line) for line in (study / 'runs.jsonl').read_text(encoding='utf-8').splitlines()]
cases = [case for case in protocol['cases'] if case['n'] in (6, 8) and case['seed'] in (6, 7)]
assert len(cases) == 8
environment = collect_environment()
print('운영체제:', environment['os'])
print('CPU:', environment['cpu'])
print('Python:', environment['python'])
print('사례 수:', len(cases))

운영체제: Linux-6.18.58.1-microsoft-standard-WSL2-x86_64-with-glibc2.41
CPU: Intel(R) Core(TM) i5-8500 CPU @ 3.00GHz
Python: 3.13.5
사례 수: 8


### 담당: 공통 평가 김대건 · 실행 코드 AI

In [2]:
from IPython.display import Markdown, display

methods = {item['method']: item for item in protocol['methods']}
selected_methods = ['betweenness', 'demand_bundle']
results = []
for case in cases:
    bundle = load_bundle(root / case['path'], require_reference=True)
    assert bundle.case_id == case['case_id']
    for method_name in selected_methods:
        method = methods[method_name]
        source = root / method['path']
        assert sha256(source.read_bytes()).hexdigest() == method['sha256']
        process = subprocess.run([sys.executable, '-I', str(source)], input=scenario_input(bundle),
                                 capture_output=True, timeout=10, check=True)
        selected = protected_ids(process.stdout, bundle)
        score = Evaluator(bundle.scenario).evaluate(list(selected), bundle.budget)
        reference = bundle.optimal_reference.evaluation
        previous = next(row for row in saved if row['name'] == case['name'] and row['method'] == method_name)
        assert previous['status'] == 'completed'
        assert (score.connected_pairs, score.delivery_units) == (previous['connected_pairs'], previous['delivery_units'])
        results.append({'case': case['name'], 'method': method_name, 'score': score, 'reference': reference})

lines = ['| 사례 | 방법 | 비용 / 예산 | 연결 쌍 | 최적 연결 쌍 | 전달률 |',
         '|---|---|---:|---:|---:|---:|']
for result in results:
    case, score, ref = result['case'], result['score'], result['reference']
    bundle = load_bundle(root / 'scenarios/problem-bank/bundles' / (case + '.json'))
    label = '매개중심성/비용' if result['method'] == 'betweenness' else '수요 경로 묶음'
    lines.append(f"| {case} | {label} | {score.cost}/{bundle.budget} | "
                 f"{score.connected_pairs} | {ref.connected_pairs} | {score.delivery_ratio:.2%} |")
display(Markdown(chr(10).join(lines)))
print('저장 점수와 재계산 결과 일치:', len(results), '건')

/home/user/os-project-26-2/protection/flow.py:187: OptimizeWarning: Unrecognized options detected: {'threads': 1}. These will be passed to HiGHS verbatim.
  result = linprog(lp.c, A_ub=lp.G, b_ub=lp.b,


| 사례 | 방법 | 비용 / 예산 | 연결 쌍 | 최적 연결 쌍 | 전달률 |
|---|---|---:|---:|---:|---:|
| n6-seed6-p20 | 매개중심성/비용 | 7/8 | 10 | 10 | 57.14% |
| n6-seed6-p20 | 수요 경로 묶음 | 7/8 | 10 | 10 | 57.14% |
| n6-seed6-p40 | 매개중심성/비용 | 12/16 | 10 | 15 | 73.81% |
| n6-seed6-p40 | 수요 경로 묶음 | 13/16 | 15 | 15 | 78.57% |
| n6-seed7-p20 | 매개중심성/비용 | 6/7 | 10 | 15 | 100.00% |
| n6-seed7-p20 | 수요 경로 묶음 | 7/7 | 15 | 15 | 59.46% |
| n6-seed7-p40 | 매개중심성/비용 | 11/15 | 10 | 15 | 100.00% |
| n6-seed7-p40 | 수요 경로 묶음 | 7/15 | 15 | 15 | 59.46% |
| n8-seed6-p20 | 매개중심성/비용 | 15/16 | 15 | 21 | 56.12% |
| n8-seed6-p20 | 수요 경로 묶음 | 16/16 | 21 | 21 | 54.08% |
| n8-seed6-p40 | 매개중심성/비용 | 31/33 | 21 | 28 | 82.65% |
| n8-seed6-p40 | 수요 경로 묶음 | 25/33 | 21 | 28 | 78.57% |
| n8-seed7-p20 | 매개중심성/비용 | 12/13 | 21 | 21 | 61.17% |
| n8-seed7-p20 | 수요 경로 묶음 | 12/13 | 21 | 21 | 66.99% |
| n8-seed7-p40 | 매개중심성/비용 | 25/26 | 21 | 28 | 61.17% |
| n8-seed7-p40 | 수요 경로 묶음 | 19/26 | 28 | 28 | 73.79% |

저장 점수와 재계산 결과 일치: 16 건


## 해석

표의 사례는 재현 검사용 8개 조건이다. 전체 92개 사례의 실행별 원자료는 `results/agent-study/2026-11-24/runs.jsonl`, 사전 고정 조건은 `protocol.json`, 집계는 `summary.json`을 확인한다. 저장된 참조는 합성 문제은행의 완전탐색 결과다. 최종 실험의 개인 기여와 실망 대표성은 별도 확인이 필요하다.